# NB_patient_360

Builds `<catalog>.gold.patient_360` (one row per patient) from silver patients, hospitals, insurance providers and aggregated lab results, per the Silver-to-Gold STTM. Full rebuild via CREATE OR REPLACE TABLE ... AS SELECT.

In [0]:
# Catalog placeholder from the STTM (<catalog>) is exposed as a widget
dbutils.widgets.text("catalog", "adb_caresync", "Catalog")
catalog = dbutils.widgets.get("catalog")

src = f"{catalog}.silver"
tgt_table = f"{catalog}.gold.patient_360"
print(f"Source schema: {src} | Target table: {tgt_table}")

In [0]:
# Full rebuild: one row per patient, LEFT JOINs from the driving table (silver.patients)
# Note: silver.insurance_providers contains duplicate insurance_provider_id rows, so it is
# deduplicated (latest update_timestamp wins) before joining to preserve the patient grain.
spark.sql(f"""
CREATE OR REPLACE TABLE {tgt_table} AS
WITH insurance AS (
  SELECT insurance_provider_id, provider_name
  FROM (
    SELECT insurance_provider_id, provider_name,
           ROW_NUMBER() OVER (PARTITION BY insurance_provider_id
                              ORDER BY update_timestamp DESC, provider_name) AS rn
    FROM {src}.insurance_providers
  ) WHERE rn = 1
),
lab_agg AS (
  SELECT patient_id,
         COUNT(*)                                                AS total_tests,
         COUNT(CASE WHEN result_status = 'Abnormal' THEN 1 END)  AS abnormal_test_count,
         COUNT(CASE WHEN result_status = 'Critical' THEN 1 END)  AS critical_test_count,
         MAX(test_date)                                          AS last_test_date
  FROM {src}.lab_results
  GROUP BY patient_id
)
SELECT
  CAST(p.patient_id AS INT)                                              AS patient_id,
  CAST(p.first_name AS STRING)                                           AS first_name,
  CAST(p.last_name AS STRING)                                            AS last_name,
  CAST(p.date_of_birth AS DATE)                                          AS date_of_birth,
  CAST(FLOOR(DATEDIFF(current_date(), CAST(p.date_of_birth AS DATE)) / 365.25) AS INT) AS age,
  CAST(p.gender AS STRING)                                               AS gender,
  CAST(p.city AS STRING)                                                 AS city,
  CAST(p.state AS STRING)                                                AS state,
  CAST(p.registered_hospital_id AS INT)                                  AS hospital_id,
  CAST(h.hospital_name AS STRING)                                        AS hospital_name,
  CAST(p.insurance_provider_id AS INT)                                   AS insurance_provider_id,
  CAST(i.provider_name AS STRING)                                        AS insurance_provider_name,
  CAST(COALESCE(l.total_tests, 0) AS INT)                                AS total_tests,
  CAST(COALESCE(l.abnormal_test_count, 0) AS INT)                        AS abnormal_test_count,
  CAST(COALESCE(l.critical_test_count, 0) AS INT)                        AS critical_test_count,
  CAST(l.last_test_date AS DATE)                                         AS last_test_date
FROM {src}.patients p
LEFT JOIN {src}.hospitals h ON p.registered_hospital_id = h.hospital_id
LEFT JOIN insurance i       ON p.insurance_provider_id = i.insurance_provider_id
LEFT JOIN lab_agg l         ON p.patient_id = l.patient_id
""")
print(f"{tgt_table} rebuilt.")

In [0]:
record_count = spark.table(tgt_table).count()
dbutils.notebook.exit(str(record_count))

In [0]:
%sql
select * from adb_caresync.gold.patient_360